In [1]:
from mava.networks.retention import MultiScaleRetention
from omegaconf import DictConfig
import jax
import jax.numpy as jnp
import copy

# jax.config.update("jax_enable_x64", True)

bsz = 16
num_agents = 4
obs_dim = 11
num_time_steps = 100
seq_len = num_agents * num_time_steps

retnet_embed_dim = 32
retnet_num_heads = 2

2025-02-26 16:23:34.215046: W external/xla/xla/service/gpu/nvptx_compiler.cc:765] The NVIDIA driver's CUDA version is 12.5 which is older than the ptxas CUDA version (12.8.61). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
/home/ruanjohn/miniconda3/envs/mava/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
memory_config = DictConfig(
    {
        "type": "rec_sable",
        "decay_scaling_factor": 0.3,
        "timestep_positional_encoding": True,
        "timestep_chunk_size": None,
    }
)

decay_kappas = 1 - jnp.exp(jnp.linspace(jnp.log(1 / 32), jnp.log(1 / 512), retnet_num_heads))
decay_kappas *= memory_config.decay_scaling_factor
decay_kappas = decay_kappas[None, :, None, None]

In [ ]:
msr = MultiScaleRetention(
    embed_dim=retnet_embed_dim,
    n_head=retnet_num_heads,
    n_agents=num_agents,
    memory_config=memory_config,
    masked=False,
    decay_scaling_factor=memory_config.decay_scaling_factor,
)

In [4]:
key = jax.random.PRNGKey(0)
key, subkey = jax.random.split(key)

obs = jax.random.normal(subkey, (bsz, seq_len, retnet_embed_dim))

# assuming no resets
dones = jnp.zeros((bsz, seq_len), dtype=bool)

init_hstate = jnp.zeros((bsz, retnet_num_heads, retnet_embed_dim//retnet_num_heads, retnet_embed_dim//retnet_num_heads))
step_counts = jnp.arange(num_time_steps)
step_counts = step_counts[None, ...].repeat(bsz, axis=0)[..., None].repeat(num_agents, axis=-1)
step_counts = step_counts.reshape(bsz, seq_len)

In [5]:
key, init_key = jax.random.split(key)
params = msr.init(
    init_key,
    obs,
    obs,
    obs,
    init_hstate,
    dones,
    step_counts,
)

In [6]:
hstate = copy.deepcopy(init_hstate)
act_output = []


# for the decoder we use the chunkwise
for step in range(num_time_steps):

    # todo: reset later
    hstate = hstate * decay_kappas
    obs_i = obs[:, step*num_agents:(step+1)*num_agents, ...]
    dones_i = dones[:, step*num_agents:(step+1)*num_agents]
    step_counts_i = step_counts[:, step*num_agents:(step+1)*num_agents]

    out, hstate = msr.apply(params, obs_i, obs_i, obs_i, hstate, step_counts_i, method="recurrent")
    act_output.append(out)

In [7]:
act_output = jnp.concatenate(act_output, axis=1)

In [8]:
act_output.shape

(16, 400, 32)

In [9]:
hstate = copy.deepcopy(init_hstate)
train_out, _ = msr.apply(params, obs, obs, obs, hstate, dones, step_counts)

In [10]:
train_out.shape

(16, 400, 32)

In [11]:
total_error = jnp.mean(jnp.abs(train_out - act_output))
total_error

Array(0.01446979, dtype=float32)

In [12]:
jnp.abs(train_out - act_output)

Array([[[0.0086738 , 0.01361195, 0.00537209, ..., 0.01265837,
         0.01494458, 0.0185911 ],
        [0.00184659, 0.00574793, 0.00607001, ..., 0.00713671,
         0.00288633, 0.00543174],
        [0.00379084, 0.00958553, 0.00881629, ..., 0.00430346,
         0.00564132, 0.01087695],
        ...,
        [0.0083628 , 0.02380694, 0.00421088, ..., 0.0130822 ,
         0.02127572, 0.01317773],
        [0.00135793, 0.01320454, 0.00358132, ..., 0.00954084,
         0.00606678, 0.01440244],
        [0.0006398 , 0.00800974, 0.00054012, ..., 0.01413986,
         0.00643541, 0.00896428]],

       [[0.00575255, 0.02500835, 0.00843301, ..., 0.0291517 ,
         0.02834679, 0.00555907],
        [0.00022464, 0.00499086, 0.00235198, ..., 0.01122253,
         0.01388149, 0.00183167],
        [0.00537135, 0.00747663, 0.0021419 , ..., 0.0096924 ,
         0.00236989, 0.00355745],
        ...,
        [0.00489878, 0.00635488, 0.02905668, ..., 0.02609481,
         0.00269354, 0.0015536 ],
        [0.0